### Setup & configuration

In [ ]:
import sys
import importlib
import json
import time
import mlflow
import pandas as pd
import torch
from IPython.display import display, Markdown, JSON, HTML
#
sys.path.append('..')

### INPUT VARIABLES

In [ ]:
WORKLOG= """
INC-99102

System monitoring detected intermittent packet loss on core switch cluster in Toronto data node TD-14.

Initial alert triggered at 03:41 AM when ICMP ping failures exceeded threshold (packet loss > 40%) for IP range 10.22.14.0/24.

NOC engineer report:
Multiple devices intermittently responding. Device SW-CORE-19 shows unstable connectivity. SSH access failed twice.

At 03:58 AM, logs show repeated ICMP timeout for 10.22.14.23 and 10.22.14.31.

A technician was dispatched but initial remote diagnostics were inconclusive due to network flapping.

On-site technician report (Alex P.):
Device SW-CORE-19 completely unresponsive to ping and console access.
Physical inspection revealed NIC card failure suspected after overheating warning logs.

Replacement procedure initiated:
- removed faulty network interface card
- installed replacement NIC module
- rebooted device

By 05:10 AM, device restored and stable ping response confirmed.

Customer impact: intermittent service degradation across VLAN 220-245.

Next action: monitor stability for 24 hours and confirm no packet loss recurrence.

"""

# Knowledge Extraction

### setup MLflow

In [37]:
import mlflow

mlflow.set_experiment("incident-log-extraction-qwen")


<Experiment: artifact_location='/media/zirox2025/DATA/running/GitHub/Kowledge-extraction/notebooks/mlruns/1', creation_time=1778954595034, experiment_id='1', last_update_time=1778954595034, lifecycle_stage='active', name='incident-log-extraction-qwen', tags={}, trace_location=None, workspace='default'>

### Instantiate the LLM model

In [ ]:
MODEL_NAME = "../model/Qwen2.5-1.5B-Instruct"

import json
import time
import os
import pandas as pd
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

# define the absolute path of the LLM model
MODEL_PATH = os.path.abspath(MODEL_NAME)
print(f"Loading model {MODEL_PATH}...")

# instantiate the model and tokenizer with local_files_only=True to ensure it loads from the local path
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    local_files_only=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
)

model.eval()

### Run the experiment

In [44]:
import importlib
import prompts

importlib.reload(prompts)

import main

importlib.reload(main)

from typing import Any, Dict, Tuple

# load modules/prompts
from prompts import PROMPT_V2 as prompt_template
from main import extract, verify, repair, evaluate_json


# define the extractor and verifier models
model_extractor = model
model_verifier = model

# RUN EXPERIMENT
start_time = time.time()

# =========================
# MLflow RUN
# =========================

with mlflow.start_run():

    mlflow.log_param("model", MODEL_NAME)
    mlflow.log_param("system", "clean_two_model_pipeline")

    start_time: float = time.time()

    # Step 1: Extract
    extracted, t_extract = extract(
        model_extractor,
        tokenizer,
        WORKLOG,
        prompt_template,
    )

    # Step 2: Verify
    verification: str = verify(
        model_verifier,
        tokenizer,
        WORKLOG,
        extracted,
    )

    # Step 3: Repair
    final_output: str = repair(
        model_extractor,
        tokenizer,
        WORKLOG,
        extracted,
        verification,
    )

    total_latency: float = time.time() - start_time

    # Metrics
    metrics: Dict[str, int] = evaluate_json(final_output)
    metrics["latency_sec"] = total_latency

    for k, v in metrics.items():
        mlflow.log_metric(k, v)

    # Artifacts
    mlflow.log_text(WORKLOG, "input.txt")
    mlflow.log_text(extracted, "extracted.json")
    mlflow.log_text(verification, "verification.json")
    mlflow.log_text(final_output, "final_output.json")

    mlflow.log_text(prompt_template, "prompt_template.txt")

    print("\n=== EXTRACTED ===\n", extracted)
    print("\n=== VERIFICATION ===\n", verification)
    print("\n=== FINAL OUTPUT ===\n", final_output)

print("\n===== ELAPSED TIME =====\n")
print(f"{total_latency:.2f} seconds")

Running extraction...


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Running verification...
Running repair...

=== EXTRACTED ===
 {
  "incident_summary": "An intermittent packet loss issue was identified on the core switch cluster in the Toronto data node TD-14. Initial alerts showed high ICMP ping failures affecting the IP range 10.22.14.0/24. Remote diagnostics were inconclusive due to network instability. A technician conducted a physical inspection and discovered a NIC card failure, leading to a replacement of the faulty card. The device returned to normal operation by 05:10 AM.", 
  "event_timeline": [
    {
      "timestamp": "03:41 AM",
      "event": "ICMP ping failures exceed threshold",
      "source": "System Monitoring"
    },
    {
      "timestamp": "03:58 AM",
      "event": "Repeated ICMP timeouts",
      "source": "Logs"
    },
    {
      "timestamp": "05:10 AM",
      "event": "Device restored and stable ping response",
      "source": "Technician Report"
    }
  ],
  "entities": {
    "devices": ["SW-CORE-19"],
    "circuits": [],
 

In [41]:
extracted

'system\nYou are a precise industrial reasoning system.\nuser\n\nYou are an industrial incident reconstruction system.\n\nYour job is NOT summarization.\n\nYour job is HIGH-FIDELITY EVENT RECONSTRUCTION from noisy operational logs.\n\nYou must preserve ALL diagnostic, temporal, and telemetry information.\n\n------------------------------------------------------------\nTASK OVERVIEW (IMPORTANT)\n------------------------------------------------------------\n\nYou will reconstruct a structured incident report with:\n\n1. Full event timeline (no compression)\n2. Complete entity extraction (devices, IPs, technicians)\n3. Conflict and uncertainty detection\n4. Final resolution with root cause fidelity\n5. Internal consistency validation (VERY IMPORTANT)\n\n------------------------------------------------------------\nOUTPUT SCHEMA (STRICT JSON ONLY)\n------------------------------------------------------------\n\n{\n  "incident_summary": "",\n  "event_timeline": [\n    {\n      "timestamp": 

In [43]:
# Extract JSON from markdown code blocks if present
if "```json" in extracted:
    result = "```json" + extracted.split("```json")[1].strip()
elif "```" in extracted:
    result = "```" + extracted.split("```")[1].strip()
result

'```json{\n  "incident_summary": "An intermittent packet loss issue was identified on the core switch cluster in the Toronto data node TD-14. Initial alerts showed high ICMP ping failures affecting IP range 10.22.14.0/24. Remote diagnostics were inconclusive due to network flapping, leading to a physical inspection revealing a NIC card failure. The faulty card was replaced, restoring stable operation by 05:10 AM. Customer impact included intermittent service degradation across VLANs 220-245.",\n  "event_timeline": [\n    {\n      "timestamp": "03:41 AM",\n      "event": "ICMP ping failures exceeding threshold (packet loss > 40%) for IP range 10.22.14.0/24"\n    },\n    {\n      "timestamp": "03:58 AM",\n      "event": "Repeated ICMP timeouts for IP addresses 10.22.14.23 and 10.22.14.31"\n    },\n    {\n      "timestamp": "05:10 AM",\n      "event": "Device SW-CORE-19 restored and stable ping response confirmed"\n    }\n  ],\n  "entities": {\n    "devices": ["SW-CORE-19"],\n    "circuit